# M1 PoC - Regresión simbolica con PySr (datos sinteticos)
Objetivo: Observar que PySR recupera una formula conocida a partir de datos generados.
**Formula objetivo:** `y = 2x² - 3x + 1`

# 0 - Imports y verificacion del entorno

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sympy as sp
from pysr import PySRRegressor
from sklearn.metrics import r2_score

print("Imports OK")
print(f"numpy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"sympy: {sp.__version__}")

Imports OK
numpy: 2.4.6
pandas: 3.0.3
sympy: 1.14.0


# 1 - Generar datos sinteticos
Usamos `y = 2x² - 3x + 1` con ruido para simular condiciones realistas

In [ ]:
# Parametros
N = 200             # Numero de muestras
SEED = 42           # Semilla aleatoria
NOISE_STD = 0.1     # Ruido gaussiano
X_RANGE = (-3, 3)   # rango de x

# Formula objetivo
TRUE_FORMULA = "y = 2x^2 - 3x + 1"

rng = np.random.default_rng(SEED)
x = rng.uniform(*X_RANGE, size=N)
y_true = 2 * x**2 - 3 * x + 1
y_noisy = y_true + rng.normal(0, NOISE_STD, size=N)

print(f"Dataset: {N} muestras, x en {X_RANGE}, ruido {NOISE_STD}")
print(f"Formula objetivo: {TRUE_FORMULA}")
print(f"y: min = {y_noisy.min():.2f}, max = {y_noisy.max():.2f}, mean = {y_noisy.mean():.2f}")

In [ ]:
# Visualizar los datos generados
x_sorted = np.sort()
y_sorted = 2 * x_sorted**2 - 3 * x_sorted + 1

fig, ax = plt.subplots(figsize=(8,4))
ax.scatter(x, y_noisy, s = 10, alpha = 0,5, label = "Datos con ruido", color = "steelblue")
ax.plot(x_sorted, y_sorted, color = "crimson", line_width = 2, label = "Formula real")

# 2 - Configurar y correr PySR
Operadores: solo `+`, `-`, `*`

In [ ]:
model = PySRRegressor(
    niterations = 40,
    binary_operators = ["+", "-", "*"],
    unary_operators = [],
    model_selection = "best",
    loss = "loss(prediction, target) = (prediction - target)**2",
    complexity_of_constants = 1,
    maxdepth = 5,
    verbosity = 1,
    random_state = SEED,
    progress = True,
)

X_fit = x.reshape(-1, 1)

print("Corriendo PySR...")
model.fit(X_fit, y_noisy, variable_names = ["x"])
print("\n Busqueda terminada")

# 3 - Evaluar Resultados

In [ ]:
# Frente de pareto
print("Frente de pareto")
print(model)

In [ ]:
# Mejorar formula segun model_selection = "best"
best_formula_sympy = model.sympy()
best_formula_simplified = sp.simplify(best_formula_sympy)

print(f"Formula objetivo: {TRUE_FORMULA}")
print(f"Formula encontrada (raw): {best_formula_sympy}")
print(f"Formula encontrada (simplificada): {best_formula_simplified}")

In [ ]:
# Metricas
y_pred = model.predict(X_fit)
r2 = r2_score(y_noisy, y_pred)
rmse = np.sqrt(np.mean((y_noisy - y_pred)**2)

print(f"R^2 = {r2:.6f} (objetivo: >= 0.99 con ruido (ruido = {NOISE_STD}")
print(f"RMSE = {rmse:.6f}")

In [ ]:
# Grafica: predicho vs real
fig, axes = plt.subplots(1, 2, figsize = (12, 4))

# Panel izquierdo: curvas
ax = axes[0]
ax.scatter(x, y_noisy, s=10, alpha=0.4, label='Datos', color='steelblue')
ax.plot(x_sorted, y_sorted, color='crimson', linewidth=2, label='Fórmula real')
ax.plot(x_sorted, model.predict(x_sorted.reshape(-1, 1)),
        color='forestgreen', linewidth=2, linestyle='--', label='PySR encontrada')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Curvas: real vs PySR')
ax.legend(fontsize=8)

# Panel derecho: predicho vs real (scatter)
ax = axes[1]
ax.scatter(y_noisy, y_pred, s=10, alpha=0.4, color='steelblue')
lims = [min(y_noisy.min(), y_pred.min()), max(y_noisy.max(), y_pred.max())]
ax.plot(lims, lims, 'k--', linewidth=1, label='Perfecto')
ax.set_xlabel('y real')
ax.set_ylabel('y predicho')
ax.set_title(f'Predicho vs Real  (R²={r2:.4f})')
ax.legend(fontsize=8)

plt.suptitle('M1 · Validación PoC — PySR', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show())

In [ ]:
# Residuales
residuals = y_noisy - y_pred

fig, ax = plt.subplots(figsize = (8, 3))
ax.scatter(x, residuals, s=10, alpha=0.4, color='steelblue')
ax.axhline(0, color='crimson', linewidth=1)
ax.set_xlabel('x')
ax.set_ylabel('Residual (y - ŷ)')
ax.set_title('Residuales — sin estructura → modelo bien ajustado')
plt.tight_layout()
plt.show()

print(f"Residuales: mean={residuals.mean():.4f}, std={residuals.std():.4f}")
print(f"(Esperado: mean≈0, std≈{NOISE_STD} — igual al ruido que agregamos)")